In [16]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
import torch
from pathlib import Path
import pandas as pd
import ast
from PIL import Image, ImageDraw, ImageFont
import math

In [2]:
from huggingface_hub import notebook_login

notebook_login()

In [17]:
COLOR_MAPPING = {
    (0xFF, 0xD7, 0x00): 'common room',
    (0xFF, 0xA5, 0x00): 'master room',
    (0xEE, 0xE8, 0xAA): 'living room',
    (0x6B, 0x8E, 0x23): 'balcony',
    (0xAD, 0xD8, 0xE6): 'bathroom',
    (0xF0, 0x80, 0x80): 'kitchen',
    (0xDD, 0xA0, 0xDD): 'storage',
    (0xDA, 0x70, 0xD6): 'dining',
}

In [ ]:
# class LocalVisionLLM:
#     def __init__(self, model_id: str, device: str = 'cuda'):
#         # Processor for both vision & text
#         self.processor = AutoProcessor.from_pretrained(
#             model_id,
#             use_auth_token=True,
#             trust_remote_code=True
#         )

#         # Vision‑language conditional generation model
#         self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
#             model_id,
#             torch_dtype=torch.float16,
#             device_map='auto',
#             trust_remote_code=True,
#             use_auth_token=True
#         )
#         # track actual device (could be spread across GPUs)
#         self.device = next(self.model.parameters()).device

#     def __call__(self, images: list[Image.Image], prompt: str, **generate_kwargs) -> str:
#         # 1) Build the “chat” messages list
#         messages = [{"type": "text",  "content": prompt}]
#         messages += [{"type": "image", "content": img} for img in images]

#         # 2) Turn messages into a single text string with the model’s chat template
#         #    (this adds any necessary <|user|>, <|assistant|> tokens and the generation prompt)
#         chat_text = self.processor.apply_chat_template(
#             messages,
#             tokenize=False,
#             add_generation_prompt=True
#         )

#         # 3) Extract visual inputs (pixel buffers, bboxes, etc.)
#         image_inputs, video_inputs = process_vision_info(messages)  # returns two things; video_inputs will be None here :contentReference[oaicite:0]{index=0}

#         # 4) Tokenize the chat text
#         text_inputs = self.processor(
#             chat_text,
#             return_tensors="pt",
#             add_special_tokens=False  # tokens already handled by apply_chat_template
#         )

#         # 5) Merge text + images into one input dict
#         #    (drop video_inputs since you’re only doing stills)
#         inputs = {**text_inputs, **(image_inputs or {})}
#         inputs = {k: v.to(self.device) for k, v in inputs.items()}

#         # 6) Generate
#         defaults = dict(max_new_tokens=256, do_sample=False)
#         outputs = self.model.generate(**inputs, **{**defaults, **generate_kwargs})

#         # 7) Decode & strip off the echoed prompt
#         full = self.processor.decode(outputs[0], skip_special_tokens=True)
#         return full[len(prompt):].strip()

In [18]:

class LocalVisionLLM:
    def __init__(self, model_id: str, device_map: str = 'auto', torch_dtype="auto"):
        # 1) Load processor
        self.processor = AutoProcessor.from_pretrained(
            model_id,
            use_auth_token=True,
            trust_remote_code=True
        )
        # 2) Load model
        self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            model_id,
            torch_dtype=torch_dtype,
            device_map=device_map,
            trust_remote_code=True,
            use_auth_token=True
        )
        # track actual device
        self.device = next(self.model.parameters()).device

    def __call__(self, images: list[Image.Image], prompt: str, max_new_tokens: int = 256, **gen_kwargs) -> str:
        # Build a single “user” message that contains both the images and the text
        messages = [
            {
                "role": "user",
                "content": [
                    *[
                        {"type": "image", "image": img}
                        for img in images
                    ],
                    {"type": "text", "text": prompt}
                ],
            }
        ]

        # 1) Format chat
        chat_text = self.processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        # 2) Extract vision inputs
        image_inputs, video_inputs = process_vision_info(messages)

        # 3) Prepare model inputs (batched)
        model_inputs = self.processor(
            text=[chat_text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        )
        model_inputs = {k: v.to(self.device) for k, v in model_inputs.items()}

        # 4) Generate
        gen_kwargs = dict(max_new_tokens=max_new_tokens, **gen_kwargs)
        generated_ids = self.model.generate(**model_inputs, **gen_kwargs)

        # 5) Trim off the prompt tokens
        #    generated_ids is shape [batch, seq_len]; inputs.input_ids is [batch, seq_len_in]
        input_ids = model_inputs["input_ids"]
        trimmed = [
            out_ids[input_ids.shape[1]:]
            for out_ids in generated_ids
        ]

        # 6) Decode
        output_texts = self.processor.batch_decode(
            trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )
        # single-example batch, so take [0]
        return output_texts[0].strip()

In [19]:
model_id = "Qwen/Qwen2.5-VL-7B-Instruct"
vlm = LocalVisionLLM(model_id)

/home/hice1/hzhang931/scratch/planscape/venv/lib/python3.11/site-packages/transformers/models/auto/processing_auto.py:243: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/home/hice1/hzhang931/scratch/planscape/venv/lib/python3.11/site-packages/transformers/modeling_utils.py:4056: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [32]:
# csv_path = Path("groups_diff_first_results.csv")
# csv_path = Path("groups_same_second_results.csv")
csv_path = Path("groups_same_first_global_results.csv")
df = pd.read_csv(csv_path)

In [27]:

# def build_prompt(options: list[str]) -> str:
#     text = (
#         "I am showing you 5 apartment floorplan images.\n"
#         "4 share the same layout and 1 is different.\n\n"
#         "Please look carefully at spatial relationships, room types, and sizes.\n"
#         "Which example is the different one, and why?\n\n"
#         "Structure your reply exactly like:\n"
#         "1. **Different example:** <ID>\n"
#         "2. **Why:** <brief reasoning>\n\n"
#         "Examples by number:\n"
#     )
#     for idx, pid in enumerate(options, start=1):
#         text += f"- Example {idx}: ID {pid}\n"
#     return text

def build_prompt(options: list[str]) -> str:
    text = (
        "I am showing you 5 apartment floorplan images.\n"
        "4 out of these descriptions share the same underlying floorplan pattern and 1 out of them has a different floorplan pattern\n\n"
        "Each of these examples come with a legend indicating the correspondence between colors and room types.\n"
        "Please look carefully at spatial relationships, room types, and sizes.\n"
        "\n"
        "Examples by number:\n"
    )
    for idx, pid in enumerate(options, start=1):
        text += f"- Example {idx}: ID {pid}\n"

    text += (
        "\nWhich example is the different one, and why?\n\n"
        "Structure your reply exactly like:\n"
        "1. **Different example:** <ID>\n"
        "2. **Why:** <brief reasoning>\n\n"
    )

    return text

In [ ]:
# # 2. iterate
# results = []
# data_dir = Path("../../data/floorplan_reoriented")

# for _, row in df.iterrows():
#     options = ast.literal_eval(row['options'])
#     prompt  = build_prompt(options)

#     # load the five floorplan images
#     images = []
#     for pid in options:
#         img_path = data_dir / f"{pid}.png"
#         images.append(Image.open(img_path).convert("RGB"))

#     # call your vision‑language LLM
#     # it handles processor, device placement, process_vision_info, generate(), decode(), and prompt‑stripping

#     answer = vl_llm(images, prompt, max_new_tokens=256, do_sample=False)

#     # parse out the “1. Different example: X” line
#     first_line = answer.splitlines()[0]
#     predicted = first_line.split(":", 1)[1].strip()

#     results.append({
#         'base_cluster':  row['base_cluster'],
#         'other_cluster': row['other_cluster'],
#         'options':       options,
#         'outlier_id':    row['outlier_id'],
#         'predicted_id':  predicted,
#         'raw_response':  answer
#     })

# # 3. save
# df_out = pd.DataFrame(results)
# df_out.to_csv("outputs/llm_image_results_qwen2.5-vl.csv", index=False)
# print(f"Wrote {len(df_out)} results to outputs/llm_image_results_qwen2.5-vl.csv")

TypeError: 'Image' object is not iterable

In [ ]:
results = []
data_dir = Path("../../data/floorplan_image")

for _, row in df.iterrows():
    options = ast.literal_eval(row['options'])
    prompt  = build_prompt(options)

    # load images
    images = [
        Image.open(data_dir / f"{pid}.png").convert("RGB")
        for pid in options
    ]

    # call the model
    answer = vlm(images, prompt, do_sample=False)

    # parse out the “1. Different example: X” line
    first_line = answer.splitlines()[0]
    predicted = first_line.split(":", 1)[1].strip()

    results.append({
        'base_cluster':  row['base_cluster'],
        'other_cluster': row['other_cluster'],
        'options':       options,
        'outlier_id':    row['outlier_id'],
        'predicted_id':  predicted,
        'raw_response':  answer
    })

# save
df_out = pd.DataFrame(results)
out_path = Path("outputs/vlm_qwen2.5-vl_diff_first.csv")
out_path.parent.mkdir(exist_ok=True)
df_out.to_csv(out_path, index=False)
print(f"Wrote {len(df_out)} results to {out_path}")

KeyboardInterrupt: 

In [ ]:
def add_legend_two_columns(image: Image.Image, mapping: dict) -> Image.Image:
    swatch = 20
    pad    = 5
    font   = ImageFont.load_default()

    entries = list(mapping.items())
    cols    = 2
    rows    = math.ceil(len(entries) / cols)
 
    leg_h = rows * (swatch + pad) + pad
    leg_w = image.width 

    legend = Image.new("RGB", (leg_w, leg_h), "white")
    draw   = ImageDraw.Draw(legend)

    col_w = leg_w // cols

    for idx, (color, label) in enumerate(entries):
        col = idx // rows
        row = idx % rows
        x0  = col*col_w + pad
        y0  = row*(swatch + pad) + pad

        draw.rectangle([x0, y0, x0+swatch, y0+swatch], fill=color)
        draw.text((x0+swatch+pad, y0), label, fill="black", font=font)

    combined = Image.new("RGB", (image.width, image.height + leg_h))
    combined.paste(image, (0, 0))
    combined.paste(legend, (0, image.height))
    return combined

In [33]:
results = []
data_dir = Path("../../data/floorplan_image")

for _, row in df.iterrows():
    options = ast.literal_eval(row['options'])
    prompt  = build_prompt(options)

    images = []

    for pid in options:
        img = Image.open(data_dir / f"{pid}.png").convert("RGB")
        img = add_legend_two_columns(img, COLOR_MAPPING)
        images.append(img)

    answer = vlm(images, prompt, do_sample=False)

    first_line = answer.splitlines()[0]
    predicted = first_line.split(":", 1)[1].strip()

    results.append({
        'base_cluster':  row['base_cluster'],
        'other_cluster': row['other_cluster'],
        'options':       options,
        'outlier_id':    row['outlier_id'],
        'predicted_id':  predicted,
        'raw_response':  answer
    })

# save
df_out = pd.DataFrame(results)
out_path = Path("outputs/vlm_qwen2.5-vl_same_first_global_wlegend.csv")
out_path.parent.mkdir(exist_ok=True)
df_out.to_csv(out_path, index=False)
print(f"Wrote {len(df_out)} results to {out_path}")

/home/hice1/hzhang931/scratch/planscape/venv/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `1e-06` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(


Wrote 100 results to outputs/vlm_qwen2.5-vl_same_first_global_wlegend.csv
